In [1]:
import pandas as pd
df = pd.read_csv("../data/raw/postings.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

Shape: (123849, 31)

Columns: ['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null   float64
 13  original_liste

In [4]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Missing Count" : missing,
    "Missing %" : missing_pct
})

missing_df[missing_df["Missing Count"]>0]
                    

,Missing Count,Missing %
closed_time,122776,99.13
skills_desc,121410,98.03
med_salary,117569,94.93
remote_allowed,108603,87.69
applies,100529,81.17
min_salary,94056,75.94
max_salary,94056,75.94
currency,87776,70.87
compensation_type,87776,70.87
pay_period,87776,70.87


In [6]:
#Drop columns that are too empty and thus cnt be used
dropcol = ['closed_time', 'skills_desc', 'med_salary', 'remote_allowed', 'applies']
df = df.drop(columns=dropcol)
#Drop rows where company is missing
df = df.dropna(subset=['company_name']) 
#Fill missing experience levels
df['formatted_experience_level'] = df['formatted_experience_level'].fillna('Not Specified')

print('New Shape:', df.shape)
print('Columns remaining:', df.columns.tolist())


New Shape: (122130, 26)
Columns remaining: ['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'min_salary', 'formatted_work_type', 'original_listed_time', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'formatted_experience_level', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']


In [9]:
print('duplicate rows:', df.duplicated().sum())
print('Unique job titles:', df['title'].nunique())
print('Unique companies:', df['company_name'].nunique())
print('Unique locations:', df['location'].nunique())
print('\nWork type breakdown')
print(df['work_type'].value_counts())
print('\nExperience level breakdown')
print(df['formatted_experience_level'].value_counts())

duplicate rows: 0
Unique job titles: 71880
Unique companies: 24428
Unique locations: 8462

Work type breakdown
work_type
FULL_TIME     97552
CONTRACT      11955
PART_TIME      9488
TEMPORARY      1176
INTERNSHIP      955
VOLUNTEER       555
OTHER           449
Name: count, dtype: int64

Experience level breakdown
formatted_experience_level
Mid-Senior level    41367
Entry level         36681
Not Specified       27948
Associate            9743
Director             3729
Internship           1449
Executive            1213
Name: count, dtype: int64


In [10]:
salary_df = df.dropna(subset=['min_salary', 'max_salary'])
print("Rows with salary data:", len(salary_df))
print("\nSalary stats:")
print(salary_df[['min_salary', 'max_salary', 'normalized_salary']].describe())

print('\nPay period breakdown:')
print(salary_df['pay_period'].value_counts())

Rows with salary data: 29338

Salary stats:
         min_salary    max_salary  normalized_salary
count  2.933800e+04  2.933800e+04       2.933800e+04
mean   6.511109e+04  9.224880e+04       2.372836e+05
std    4.997555e+05  7.064466e+05       5.652021e+06
min    1.000000e+00  1.000000e+00       1.000000e+00
25%    3.750000e+01  4.867500e+01       5.950000e+04
50%    6.020000e+04  8.000000e+04       9.000000e+04
75%    1.000000e+05  1.400000e+05       1.305000e+05
max    8.500000e+07  1.200000e+08       5.356000e+08

Pay period breakdown:
pay_period
YEARLY      18825
HOURLY      10051
MONTHLY       276
WEEKLY        177
BIWEEKLY        9
Name: count, dtype: int64


In [13]:
#CleaningSalaries
#Keep only Yearly salaries and remove outliers
salary_clean = salary_df[
    (salary_df["pay_period"] == 'YEARLY') &
    (salary_df["min_salary"] >= 10000) &
    (salary_df["max_salary"] <= 500000)
].copy()

print("Clean salary rows:", len(salary_clean))
print("\nCleaned salary stas:")
print(salary_clean[['min_salary', 'max_salary']].describe().round(0))

Clean salary rows: 18329

Cleaned salary stas:
       min_salary  max_salary
count     18329.0     18329.0
mean      97753.0    136257.0
std       45935.0     67669.0
min       10000.0     12000.0
25%       65000.0     85000.0
50%       89400.0    120000.0
75%      120000.0    170000.0
max      450000.0    500000.0


In [15]:
df.to_csv("../data/exports/postings_cleaned.csv", index=False)
print("Cleaned file saved to data/exports/posting_cleaned.csv")
print("Final shape:", df.shape)

Cleaned file saved to data/exports/posting_cleaned.csv
Final shape: (122130, 26)
